In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/kobe-bryant-shot-selection/sample_submission.csv.zip
/kaggle/input/kobe-bryant-shot-selection/data.csv.zip


使用 unzip 命令（最简洁）

In [ ]:
!unzip /kaggle/input/kobe-bryant-shot-selection/data.csv.zip -d /kaggle/working/

Archive:  /kaggle/input/kobe-bryant-shot-selection/data.csv.zip
replace /kaggle/working/data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

-d /kaggle/working/ 表示把文件解压到当前工作目录（可改为其他路径）。

解压后，你可以用 ls /kaggle/working/ 查看结果。

In [33]:
ls /kaggle/working/

data.csv  submission1.csv  submission2.csv  submission3.csv  submission.csv


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
filename="/kaggle/working/data.csv"
df=pd.read_csv(filename)
print("df:",df)

df:              action_type combined_shot_type  game_event_id   game_id      lat  \
0              Jump Shot          Jump Shot             10  20000012  33.9723   
1              Jump Shot          Jump Shot             12  20000012  34.0443   
2              Jump Shot          Jump Shot             35  20000012  33.9093   
3              Jump Shot          Jump Shot             43  20000012  33.8693   
4      Driving Dunk Shot               Dunk            155  20000012  34.0443   
...                  ...                ...            ...       ...      ...   
30692          Jump Shot          Jump Shot            397  49900088  33.9963   
30693           Tip Shot           Tip Shot            398  49900088  34.0443   
30694  Running Jump Shot          Jump Shot            426  49900088  33.8783   
30695          Jump Shot          Jump Shot            448  49900088  33.7773   
30696          Jump Shot          Jump Shot            471  49900088  33.9723   

       loc_x  loc_y    

In [5]:
print("df['season']:",df["season"])

df['season']: 0        2000-01
1        2000-01
2        2000-01
3        2000-01
4        2000-01
          ...   
30692    1999-00
30693    1999-00
30694    1999-00
30695    1999-00
30696    1999-00
Name: season, Length: 30697, dtype: object


In [6]:
df['dist']=np.sqrt(df['loc_x']**2+df['loc_y']**2)
loc_x_zero=df['loc_x']==0
df['angle']=np.array([0]*len(df))
df['angle']=df['angle'].astype(float)
df.loc[~loc_x_zero,'angle']=np.arctan(df.loc[~loc_x_zero,'loc_y']/df.loc[~loc_x_zero,'loc_x'])
df.loc[loc_x_zero,'angle']=np.pi/2
df['remaining_time']=df['minutes_remaining']*60+df['seconds_remaining']
df['season_start_year']=df.season.str.split('-').str[0]
df['season_start_year']=df['season_start_year'].astype(int)
action_map={action:i for i,action in enumerate(df.action_type.unique())}
print("action_map:",action_map)
df['action_type_enumerated']=df.action_type.map(action_map)

action_map: {'Jump Shot': 0, 'Driving Dunk Shot': 1, 'Layup Shot': 2, 'Running Jump Shot': 3, 'Driving Layup Shot': 4, 'Reverse Layup Shot': 5, 'Reverse Dunk Shot': 6, 'Slam Dunk Shot': 7, 'Turnaround Jump Shot': 8, 'Tip Shot': 9, 'Running Hook Shot': 10, 'Alley Oop Dunk Shot': 11, 'Dunk Shot': 12, 'Alley Oop Layup shot': 13, 'Running Dunk Shot': 14, 'Driving Finger Roll Shot': 15, 'Running Layup Shot': 16, 'Finger Roll Shot': 17, 'Fadeaway Jump Shot': 18, 'Follow Up Dunk Shot': 19, 'Hook Shot': 20, 'Turnaround Hook Shot': 21, 'Running Tip Shot': 22, 'Jump Hook Shot': 23, 'Running Finger Roll Shot': 24, 'Jump Bank Shot': 25, 'Turnaround Finger Roll Shot': 26, 'Hook Bank Shot': 27, 'Driving Hook Shot': 28, 'Running Reverse Layup Shot': 29, 'Driving Finger Roll Layup Shot': 30, 'Fadeaway Bank shot': 31, 'Pullup Jump shot': 32, 'Finger Roll Layup Shot': 33, 'Turnaround Fadeaway shot': 34, 'Driving Reverse Layup Shot': 35, 'Driving Slam Dunk Shot': 36, 'Step Back Jump shot': 37, 'Reverse S

In [7]:
def sort_encode(df,field):
    ct=pd.crosstab(df.shot_made_flag,df[field]).apply(lambda x:x/x.sum(),axis=0)
    temp=list(zip(ct.values[1,:],ct.columns))
    temp.sort()
    new_map={}
    for index,(acc,old_number) in enumerate(temp):
        new_map[old_number]=index
    new_field=field+'_sort_enumerated'
    df[new_field]=df[field].map(new_map)
sort_encode(df,'action_type_enumerated')
opponent_map={opp:i for i,opp in enumerate(df.opponent.unique())}
df['opponent_enumerated']=df.opponent.map(opponent_map)
sort_encode(df,'opponent_enumerated')
df['away']=df.matchup.str.contains('@')
data=df[['action_type_enumerated','shot_distance','shot_made_flag','away']].dropna()

In [8]:
print("data:",data)

data:        action_type_enumerated  shot_distance  shot_made_flag   away
1                           0             15             0.0   True
2                           0             16             1.0   True
3                           0             22             0.0   True
4                           1              0             1.0   True
5                           0             14             0.0   True
...                       ...            ...             ...    ...
30691                       4              0             0.0  False
30692                       0              4             0.0  False
30694                       3             21             1.0  False
30695                       0             26             0.0  False
30696                       0              7             0.0  False

[25697 rows x 4 columns]


In [9]:
rfc=RandomForestClassifier(random_state=2024)
xgb_clf=xgb.XGBClassifier(random_state=2024)
train=df.loc[~df.shot_made_flag.isnull(),['action_type_enumerated_sort_enumerated','shot_made_flag','away','dist',
                                         'angle','opponent_enumerated_sort_enumerated','season_start_year']]
test=df.loc[df.shot_made_flag.isnull(),['action_type_enumerated_sort_enumerated','shot_id','away','dist',
                                         'angle','opponent_enumerated_sort_enumerated','season_start_year']]
print("train:",train)
print("test:",test)

train:        action_type_enumerated_sort_enumerated  shot_made_flag   away  \
1                                         2.0             0.0   True   
2                                         2.0             1.0   True   
3                                         2.0             0.0   True   
4                                        47.0             1.0   True   
5                                         2.0             0.0   True   
...                                       ...             ...    ...   
30691                                    31.0             0.0  False   
30692                                     2.0             0.0  False   
30694                                    33.0             1.0  False   
30695                                     2.0             0.0  False   
30696                                     2.0             0.0  False   

             dist     angle  opponent_enumerated_sort_enumerated  \
1      157.000000 -0.000000                                 

In [10]:
mode=test.action_type_enumerated_sort_enumerated.mode()[0]
test.action_type_enumerated_sort_enumerated.fillna(mode, inplace=True)
clf1=xgb.XGBClassifier(random_state=2024,
                       max_depth=3,
                       learning_rate=0.3,
                       subsample=0.9498268085913895,
                       colsample_bytree=0.8,
                       min_child_weight=2,
                       gamma=0.15,
                       reg_alpha=0.2,
                       reg_lambda=0.15)
clf2=RandomForestClassifier(bootstrap=False,max_depth=10,max_features=0.1,n_estimators=250,
                            random_state=2024)
clf1.fit(train.drop('shot_made_flag',axis=1),train.shot_made_flag)
clf2.fit(train.drop('shot_made_flag',axis=1),train.shot_made_flag)

/tmp/ipykernel_40/3783277880.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test.action_type_enumerated_sort_enumerated.fillna(mode, inplace=True)


RandomForestClassifier(bootstrap=False, max_depth=10, max_features=0.1,
                       n_estimators=250, random_state=2024)

In [13]:
predictions1=clf1.predict_proba(test.drop('shot_id',axis=1))
predictions2=clf2.predict_proba(test.drop('shot_id',axis=1))
submission1=pd.DataFrame({'shot_id':test.shot_id,
                           'shot_made_flag':predictions1[:,1]})
submission2=pd.DataFrame({'shot_id':test.shot_id,
                           'shot_made_flag':predictions2[:,1]})
submission1[['shot_id','shot_made_flag']].to_csv('submission1.csv',index=False)
submission2[['shot_id','shot_made_flag']].to_csv('submission2.csv',index=False)

In [14]:
submission1=pd.read_csv('submission1.csv')
submission2=pd.read_csv('submission2.csv')
weight=0.5
submission=(submission1['shot_made_flag']*weight)+(submission2['shot_made_flag']*weight)
submission_df=pd.DataFrame({'shot_id':submission1['shot_id'],'shot_made_flag':submission})
submission_df.to_csv('submission.csv',index=False)

In [16]:
submission=pd.read_csv('submission.csv')
print("submission:",submission)

submission:        shot_id  shot_made_flag
0            2        0.409347
1            3        0.374349
2            4        0.420153
3            5        0.928929
4            6        0.386483
...        ...             ...
25692    30692        0.643013
25693    30693        0.225800
25694    30695        0.871815
25695    30696        0.260590
25696    30697        0.253792

[25697 rows x 2 columns]


XGBoost详解（原理篇） https://blog.csdn.net/weixin_55073640/article/details/129519870

一、XGBoost简介
XGBoost全称为eXtreme Gradient Boosting，即极致梯度提升树。

XGBoost是Boosting算法的其中一种，Boosting算法的思想是将许多弱分类器集成在一起，形成一个强分类器（个体学习器间存在强依赖关系，必须串行生成的序列化方法）。

XGBoost是一种提升树模型，即它将许多树模型集成在一起，形成一个很强的分类器。其中所用到的树模型则是CART回归树模型。

二、XGBoost原理

1、基本组成元素
       XGBoost的基本组成元素是：决策树。

       这些决策树即为“弱学习器”，它们共同组成了XGBoost；

       并且这些组成XGBoost的决策树之间是有先后顺序的：后一棵决策树的生成会考虑前一棵决策树的预测结果，即将前一棵决策树的偏差考虑在内，使得先前决策树做错的训练样本在后续受到更多的关注，然后基于调整后的样本分布来训练下一棵决策树。

2、整体思路
（1）训练过程——构建XGBoost模型       
       从目标函数出发，可以推导出“每个叶子节点应该赋予的权值”，”分裂节点后的信息增益“，以及”特征值重要性排序函数“。

       与之前决策树的建立方法类似。当前决策树的建立首先根据贪心算法进行划分，通过计算目标函数增益（及上面所说的”分裂节点后的信息增益“），选择该结点使用哪个特征。

       选择好哪个特征后，就要确定分左右子树的条件了（比如选择特征A，条件是A<7）：为了提高算法效率（不用一个一个特征值去试），使用“加权分位法”，计算分裂点（这里由”特征值重要性排序函数“得出分裂点）。

      并且对应叶子节点的权值就由上述的“每个叶子节点应该赋予的权值”给出。

      不断进行上述算法，直至所有特征都被使用或者已经达到限定的层数，则完整的决策树构建完成。

（2）测试过程
      将输入的特征，依次输入进XGBoost的每棵决策树。每棵决策树的相应节点都有对应的预测权值w，将“在每一棵决策树中的预测权值”全部相加，即得到最后预测结果，看谁大，谁大谁是最后的预测结果。


kobe_shot.py 代码转换运行

In [18]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

In [19]:
filename="/kaggle/working/data.csv"
df=pd.read_csv(filename)
df['dist']=np.sqrt(df['loc_x']**2+df['loc_y']**2)
loc_x_zero=df['loc_x']==0
print("loc_x_zero:",loc_x_zero)

loc_x_zero: 0        False
1        False
2        False
3        False
4         True
         ...  
30692    False
30693     True
30694    False
30695    False
30696    False
Name: loc_x, Length: 30697, dtype: bool


In [20]:
df['angle']=np.array([0]*len(df))
df['angle']=df['angle'].astype(float)
df.loc[~loc_x_zero,'angle']=np.arctan(df.loc[~loc_x_zero,'loc_y']/df.loc[~loc_x_zero,'loc_x'])
df.loc[loc_x_zero,'angle']=np.pi/2
df['season_start_year']=df.season.str.split('-').str[0]
df['season_start_year']=df['season_start_year'].astype(int)
action_map={action:i for i,action in enumerate(df.action_type.unique())}
print("action_map:",action_map)

action_map: {'Jump Shot': 0, 'Driving Dunk Shot': 1, 'Layup Shot': 2, 'Running Jump Shot': 3, 'Driving Layup Shot': 4, 'Reverse Layup Shot': 5, 'Reverse Dunk Shot': 6, 'Slam Dunk Shot': 7, 'Turnaround Jump Shot': 8, 'Tip Shot': 9, 'Running Hook Shot': 10, 'Alley Oop Dunk Shot': 11, 'Dunk Shot': 12, 'Alley Oop Layup shot': 13, 'Running Dunk Shot': 14, 'Driving Finger Roll Shot': 15, 'Running Layup Shot': 16, 'Finger Roll Shot': 17, 'Fadeaway Jump Shot': 18, 'Follow Up Dunk Shot': 19, 'Hook Shot': 20, 'Turnaround Hook Shot': 21, 'Running Tip Shot': 22, 'Jump Hook Shot': 23, 'Running Finger Roll Shot': 24, 'Jump Bank Shot': 25, 'Turnaround Finger Roll Shot': 26, 'Hook Bank Shot': 27, 'Driving Hook Shot': 28, 'Running Reverse Layup Shot': 29, 'Driving Finger Roll Layup Shot': 30, 'Fadeaway Bank shot': 31, 'Pullup Jump shot': 32, 'Finger Roll Layup Shot': 33, 'Turnaround Fadeaway shot': 34, 'Driving Reverse Layup Shot': 35, 'Driving Slam Dunk Shot': 36, 'Step Back Jump shot': 37, 'Reverse S

In [22]:
df['action_type_enumerated']=df.action_type.map(action_map)
print("df['action_type_enumerated']:",df['action_type_enumerated'])

df['action_type_enumerated']: 0        0
1        0
2        0
3        0
4        1
        ..
30692    0
30693    9
30694    3
30695    0
30696    0
Name: action_type_enumerated, Length: 30697, dtype: int64


In [23]:
def sort_encode(df,field):
    ct=pd.crosstab(df.shot_made_flag,df[field]).apply(lambda x:x/x.sum(),axis=0)
    print("ct:",ct)
    temp=list(zip(ct.values[1,:],ct.columns))
    print("temp:",temp)
    temp.sort()
    new_map={}
    for index,(acc,old_number) in enumerate(temp):
        new_map[old_number]=index
    print("new_map:",new_map)
    new_field=field+'_sort_enumerated'
    print("new_field:",new_field)
    df[new_field]=df[field].map(new_map)
sort_encode(df,'action_type_enumerated')
print("df:",df)
print("df['action_type_enumerated_sort_enumerated']:",df['action_type_enumerated_sort_enumerated'])

ct: action_type_enumerated        0         1        2         3       4   \
shot_made_flag                                                          
0.0                     0.673087  0.023346  0.61467  0.252888  0.2586   
1.0                     0.326913  0.976654  0.38533  0.747112  0.7414   

action_type_enumerated        5         6         7         8         9   ...  \
shot_made_flag                                                            ...   
0.0                     0.363363  0.081967  0.017964  0.401796  0.649007  ...   
1.0                     0.636637  0.918033  0.982036  0.598204  0.350993  ...   

action_type_enumerated        45        46        47        48   49        50  \
shot_made_flag                                                                  
0.0                     0.333333  0.565217  0.333333  0.454545  0.0  0.333333   
1.0                     0.666667  0.434783  0.666667  0.545455  1.0  0.666667   

action_type_enumerated        51        52   53   54 

In [24]:
opponent_map={opp:i for i,opp in enumerate(df.opponent.unique())}
df['opponent_enumerated']=df.opponent.map(opponent_map)
sort_encode(df,'opponent_enumerated')

ct: opponent_enumerated       0         1         2         3         4   \
shot_made_flag                                                         
0.0                  0.53483  0.555735  0.529412  0.539106  0.565404   
1.0                  0.46517  0.444265  0.470588  0.460894  0.434596   

opponent_enumerated        5        6         7         8         9   ...  \
shot_made_flag                                                        ...   
0.0                  0.563492  0.54216  0.534717  0.569767  0.535433  ...   
1.0                  0.436508  0.45784  0.465283  0.430233  0.464567  ...   

opponent_enumerated     23        24       25        26        27        28  \
shot_made_flag                                                                
0.0                  0.564  0.572854  0.56457  0.547945  0.549943  0.588761   
1.0                  0.436  0.427146  0.43543  0.452055  0.450057  0.411239   

opponent_enumerated        29        30        31   32  
shot_made_flag          

In [25]:
df['away']= df.matchup.str.contains('@')
print("df['away']:", df['away'])

df['away']: 0         True
1         True
2         True
3         True
4         True
         ...  
30692    False
30693    False
30694    False
30695    False
30696    False
Name: away, Length: 30697, dtype: bool


In [26]:
data=df[['action_type_enumerated','shot_distance','shot_made_flag','away']].dropna()
rfc=RandomForestClassifier(random_state=2024)
train=df.loc[~df.shot_made_flag.isnull(),['action_type_enumerated_sort_enumerated','shot_made_flag','away','dist','angle']]
test=df.loc[df.shot_made_flag.isnull(),['action_type_enumerated_sort_enumerated','shot_id','away','dist','angle']]
mode=test.action_type_enumerated_sort_enumerated.mode()[0]
print("mode:",mode)
test.action_type_enumerated_sort_enumerated.fillna(mode,inplace=True)


mode: 2.0


/tmp/ipykernel_40/3352436889.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test.action_type_enumerated_sort_enumerated.fillna(mode,inplace=True)


In [ ]:
# param_grid={
#     'n_estimators':np.arange(10,191,20),
#     'bootstrap':[True,False],
#     'max_depth':np.arange(3,10),
#     'max_features':np.arange(0.1,1,0.2)
# }

In [28]:
param_grid={
    'n_estimators':np.arange(10,31,20),
    'bootstrap':[True,False],
    'max_depth':np.arange(3,4),
    'max_features':np.arange(0.1,0.3,0.2)
}
grid_search=GridSearchCV(estimator=rfc, param_grid=param_grid, cv=5)
grid_search.fit(train.drop('shot_made_flag',axis=1), train.shot_made_flag)
print("输出最佳参数组合和对应的得分：")
print(grid_search.best_params_, grid_search.best_score_, grid_search.best_estimator_, grid_search.best_index_)

输出最佳参数组合和对应的得分：
{'bootstrap': True, 'max_depth': 3, 'max_features': 0.1, 'n_estimators': 30} 0.6801177612565239 RandomForestClassifier(max_depth=3, max_features=0.1, n_estimators=30,
                       random_state=2024) 1


In [29]:
clf=RandomForestClassifier(bootstrap=False, max_depth=7, max_features=0.1, n_estimators=70, random_state=2024)
clf.fit(train.drop('shot_made_flag', axis=1), train.shot_made_flag)
predictions=clf.predict_proba(test.drop('shot_id',axis=1))
submission=pd.DataFrame({'shot_id':test.shot_id, 'shot_made_flag':predictions[:,1]})
submission[['shot_id','shot_made_flag']].to_csv('submission3.csv',index=False)